# 02 — Adaptive Inference Attack

A company stores user data as embeddings in a vector database. An attacker steals one embedding — that is all they have.

Embeddings cannot be reversed directly. But the attacker can **probe the model**: generate candidate sentences, encode them, and measure how close each guess is to the stolen vector.

This notebook covers two strategies:
- **Part I — Adaptive Probing**: craft hypotheses step by step, narrowing in on the original content
- **Part II — Nearest-Neighbor Reconstruction**: search a large corpus automatically and select the most informative results

**Prerequisite** : run `00_1`, `00_2`, and `01` first.

**Setup.** Run this cell. `TARGET_ID` selects the stolen embedding. `TARGET_POSSIBLE_NAMES` is the candidate name pool used in Step 1.

In [ ]:
import gc
import faiss
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer, util

torch.set_num_threads(1)  # M1 Accelerate/OpenMP segfault fix — must be before any model load

TARGET_FILE  = 'data/target_sentences.txt'
NAMES_FILE   = 'data/names_data_&_ai.csv'
MODEL_NAME   = 'all-mpnet-base-v2'
TARGET_ID    = 'OCTO_01'

TARGET_POSSIBLE_NAMES = pd.read_csv(NAMES_FILE)['Prénom'].dropna().unique().tolist()

**Helpers.** `calculate_similarities` encodes a batch of probes and returns them sorted by cosine similarity against the target. `display_similarities` prints the ranked results.

In [ ]:
def calculate_similarities(probes: list[str]) -> list[tuple[str, float]]:
    embs = model.encode(
        probes,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=len(probes) > 100,
    )
    sims = util.cos_sim(embs, target_emb).flatten().tolist()
    return sorted(zip(probes, sims), key=lambda x: -x[1])

def display_similarities(ranked: list[tuple[str, float]]) -> None:
    for probe, sim in ranked:
        print(f'{sim:.4f} | {probe}')

**Load the target.** Parse the target file and extract the sentence assigned to `TARGET_ID`. After encoding it below, treat it as unknown — work only from its embedding vector.

In [ ]:
targets = {}
for line in Path(TARGET_FILE).read_text().splitlines():
    if ':' in line:
        tid, sentence = line.split(':', 1)
        targets[tid.strip()] = sentence.strip()

target_sentence = targets[TARGET_ID]

**Load the model.** This is the model you identified in notebook 01.

In [ ]:
model = SentenceTransformer(MODEL_NAME, device='cpu')

**Encode.** Produce the stolen embedding vector. After this cell, stop using `target_sentence` — you are now working from `target_emb` alone.

In [ ]:
target_emb = model.encode(
    [target_sentence],
    normalize_embeddings=True,
    convert_to_numpy=True,
)

## Part I — Adaptive Probing

You have the embedding vector. Each step below proposes a set of candidate sentences. Encode them, rank by similarity, and use the scores to steer the next probe.

**Step 1 — Name.** Test every candidate name against the target and identify the most likely one.

In [ ]:
...

inferred_name = ...

**Step 2 — Identity phrasing.** Test different ways of stating the inferred name. The highest-scoring formulation becomes your `IDENTITY_PREFIX` for the rest of the attack.

In [ ]:
identity_probes = [
    f"I'm {inferred_name}",
    f'I am {inferred_name}',
    f'My name is {inferred_name}',
    f"My name's {inferred_name}",
    f'This is {inferred_name}',
    f'It is {inferred_name}',
    f"I'm called {inferred_name}",
    f'People call me {inferred_name}',
    f'You can call me {inferred_name}',
    f'Call me {inferred_name}',
    f"The name's {inferred_name}",
    f'{inferred_name} here',
    f'This is {inferred_name} speaking',
    f'Hi, I\'m {inferred_name}',
    f'Hello, I\'m {inferred_name}',
    f'Hey, I\'m {inferred_name}',
    f'{inferred_name}.'
]

display_similarities(calculate_similarities(identity_probes))

**Step 3 — Category.** With `IDENTITY_PREFIX` fixed, probe broad life themes. Identify the category that scores highest — this tells you the subject of the original sentence.

In [ ]:
IDENTITY_PREFIX = 'Hey, I\'m Nicolas'

category_probes = [
    f'{IDENTITY_PREFIX}. I like video games',
    f'{IDENTITY_PREFIX}. I like music and singing',
    f'{IDENTITY_PREFIX}. I like sports and football',
    f'{IDENTITY_PREFIX}. I like reading books',
    f'{IDENTITY_PREFIX}. I work at a company',
    f'{IDENTITY_PREFIX}. I study at university',
    f'{IDENTITY_PREFIX}. I enjoy cooking and food',
    f'{IDENTITY_PREFIX}. I like movies and television',
    f'{IDENTITY_PREFIX}. I like art and photography',
    f'{IDENTITY_PREFIX}. I enjoy writing and journaling',
    f'{IDENTITY_PREFIX}. I like science and learning',
    f'{IDENTITY_PREFIX}. I work with computers and technology',
    f'{IDENTITY_PREFIX}. I spend a lot of time online',
    f'{IDENTITY_PREFIX}. I am interested in hacking and security',
    f'{IDENTITY_PREFIX}. I like building things and fixing things',
    f'{IDENTITY_PREFIX}. I enjoy outdoor activities and nature',
    f'{IDENTITY_PREFIX}. I like animals and pets',
    f'{IDENTITY_PREFIX}. I care about health and fitness',
    f'{IDENTITY_PREFIX}. I like fashion and style',
    f'{IDENTITY_PREFIX}. I like taking the train',
    f'{IDENTITY_PREFIX}. I like quiet places and staying home',
    f'{IDENTITY_PREFIX}. I enjoy meeting people and going out',
    f'{IDENTITY_PREFIX}. I have an interesting family history',
    f'{IDENTITY_PREFIX}. I care a lot about my friends and family',
    f'{IDENTITY_PREFIX}. I have unusual habits',
    f'{IDENTITY_PREFIX}. I enjoy humor and making people laugh',
    f'{IDENTITY_PREFIX}. I am interested in politics and society',
    f'{IDENTITY_PREFIX}. I think a lot about money and work',
    f'{IDENTITY_PREFIX}. I enjoy languages and culture',
    f'{IDENTITY_PREFIX}. I prefer to be alone',
]

display_similarities(calculate_similarities(category_probes))

**Step 4 — Refine.** Explore variants within the top category. Add specific details that could describe the person's habits or activities.

In [ ]:
BASE_TARGET_SENTENCE = 'Nicolas here. I like taking the train'

refined_round_1 = [
    BASE_TARGET_SENTENCE,
    f'{IDENTITY_PREFIX}. I prefer to travel alone',
    f'{IDENTITY_PREFIX}. I like being alone when I travel',
    f'{IDENTITY_PREFIX}. I enjoy spending time with friends',
    f'{IDENTITY_PREFIX}. I like listening to music in the evening',
    f'{IDENTITY_PREFIX}. I enjoy reading in my free time',
    f'{IDENTITY_PREFIX}. I like cooking simple meals at home',
    f'{IDENTITY_PREFIX}. I spend a lot of time online',
    f'{IDENTITY_PREFIX}. I am interested in technology',
    f'{IDENTITY_PREFIX}. I like learning new things',
    f'{IDENTITY_PREFIX}. I enjoy quiet weekends at home',
    f'{IDENTITY_PREFIX}. I like going out and exploring new places',
    f'{IDENTITY_PREFIX}. I care a lot about my family',
    f'{IDENTITY_PREFIX}. I enjoy watching films and series',
    f'{IDENTITY_PREFIX}. I like staying active and healthy',
    f'{IDENTITY_PREFIX}. I enjoy talking to new people',
    f'{IDENTITY_PREFIX}. I like working on personal projects',
    f'{IDENTITY_PREFIX}. I enjoy having time to myself',
]

display_similarities(calculate_similarities(refined_round_1))

**Round 2.** The top sentence from the previous step is your new baseline. Narrow further with more specific variants.

In [ ]:
refined_round_2 = [
    f'{IDENTITY_PREFIX}. I like hacking and trains',
    f'{IDENTITY_PREFIX}. I hack train systems',
    f'{IDENTITY_PREFIX}. I always travel alone in a compartment',
    f'{IDENTITY_PREFIX}. I like to book an entire train car for myself',
    f'{IDENTITY_PREFIX}. I book trains to be alone',
]

display_similarities(calculate_similarities(refined_round_2))

**Round 3.** etc.

## Part II — Nearest-Neighbor Reconstruction

Probing by hand is slow. If the attacker has access to a training corpus (leaked or self-generated), they can search it automatically as a candidate pool.

**Load the corpus** built in `00_1`.

In [ ]:
train_df    = pd.read_parquet('data/sentences_train_text_db.parquet')
train_index = faiss.read_index('data/sentences_train_vector_db.index')
print(f'{len(train_df):,} sentences | dim={train_index.d}')

**Search and diversify.** Retrieve the nearest neighbors from the index. The top results are often near-duplicates — apply a greedy diversity criterion to spread selections across different content areas. Print the results prefixed with `IDENTITY_PREFIX`.

In [ ]:
n_results = 5
n_pool    = 100

distances, indices = train_index.search(target_emb.reshape(1, -1), n_pool)
pool_sims    = distances[0]
pool_indices = indices[0]
pool_embs    = np.vstack([train_index.reconstruct(int(i)) for i in pool_indices])

selected = [0]
...

In [ ]:
for pos in selected:
    text    = train_df.iloc[pool_indices[pos]]['text']
    subject = text.split('. ', 1)[-1]
    print(f'{pool_sims[pos]:.4f} | {IDENTITY_PREFIX}. {subject}')